# 🚀 MDAAR v2.0: Multi-role Dynamic Architecture for Automated Repositories
### المنظومة الهندسية الشاملة لتوثيق المشاريع واسترجاع المعرفة بالذكاء الاصطناعي
---
**مميزات الإصدار الثاني (v2.0.0):**
1. **قراءة 100% من ملفات المشروع بلا حدود للسياق:** فحص شامل لكل ملف وقطعة برمجية (`Map-Reduce`) مع استبعاد تلقائي لـ `vendor` و`node_modules` وتعقيم الأسرار (`SecretSanitizer`).
2. **معمارية النموذج المزدوج الهجينة (Planner + Executor):**
   - **إعدادات مودل الخطة (Planner):** يضع الخطة الهندسية الشاملة ويراجع الجودة والاتساق.
   - **إعدادات مودل التنفيذ (Executor):** يفحص الملفات بالكامل ويكتب جميع حزم التوثيق (أو يمكنك استخدام نموذج واحد لكليهما).
3. **تحليل حتمي متعدد اللغات والأطر (Polyglot Analyzers):** دعم PHP (Yii2, Laravel, Symfony)، Python (AST)، JS/TS، Java/Kotlin، C#/.NET، Go، Ruby، Rust، Flutter، بالإضافة إلى SQL/Migrations و`Postman` و`OpenAPI`.
4. **مهارات مدمجة داخل المستودع لمنع الهلوسة:** تحميل مهارات التوثيق وبروتوكول `[TBD]` و`[⚠️ BLIND SPOT]` تلقائياً لتوجيه النماذج المحلية بدقة احترافية.
5. **بوابة جودة آلية (Quality Gate):** فحص صحة مخططات `Mermaid` والتحقق من الأسماء الحقيقية وتحديث `Document-Register.md` (`L0` ➔ `L1`).
6. **محرك RAG هجين دائم في `.rag_index/`:** يفهرس الوثائق وحقائق الكود والجداول والـ APIs مع تطبيع العربية وتخصيص الجمهور (`Developer`, `Management`, `End-User`).

## 1️⃣ تجهيز البيئة وتثبيت MDAAR v2.0 وربط Google Drive
ربط مساحة التخزين الدائمة وتثبيت حزمة **MDAAR** والمهارات المدمجة وأدوات تشغيل النماذج المحلية (`transformers`, `accelerate`, `bitsandbytes`).

In [ ]:
#@title 1. تفعيل الربط وتثبيت محرك MDAAR v2.0 والمكتبات { display-mode: "form" }
import os, sys, json, shutil, subprocess
from pathlib import Path

# 1. ربط Google Drive للاستدامة
DRIVE_MOUNT_POINT = '/content/drive'
if not os.path.exists(DRIVE_MOUNT_POINT):
    try:
        from google.colab import drive
        drive.mount(DRIVE_MOUNT_POINT)
        print('✅ تم ربط Google Drive بنجاح.')
    except Exception as e:
        print('⚠️ تنبيه: لم يتم ربط الدرايف تفاعلياً، سيتم استخدام مسار تخزين محلي:', e)

# 2. تثبيت مكتبة MDAAR v2.0 مع المهارات وأدوات التكميم 4-bit (T4 GPU)
print('⏳ جاري تثبيت حزمة MDAAR والاعتماديات ومكتبات التكميم (transformers, accelerate, bitsandbytes)...')
subprocess.run(['pip', 'install', '-q', '--upgrade', 'git+https://github.com/braadiv/MDAAR.git'], check=True)
subprocess.run(['pip', 'install', '-q', '--upgrade', 'transformers', 'accelerate', 'bitsandbytes'], check=True)

# 3. التحقق من cloudflared لنفق المعاينة
cf_check = subprocess.run(['which', 'cloudflared'], capture_output=True, text=True)
if not cf_check.stdout.strip():
    print('⏳ تثبيت cloudflared...')
    subprocess.run(['wget', '-q', 'https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb'], check=True)
    subprocess.run(['dpkg', '-i', 'cloudflared-linux-amd64.deb'], check=True)

from mdaar_pipeline import __version__, SkillLoader
skills_count = len(SkillLoader().list_bundled_skills())
print(f'🎉 تم تجهيز MDAAR v{__version__} وتثبيت {skills_count} مهارات مدمجة بنجاح!')

## 2️⃣ ضبط إعدادات المشروع ومستودع الكود (Source Repo)
حدد اسم المشروع ورابط مستودع الكود المصدري وطريقة المصادقة.

In [ ]:
#@title 2. ضبط مستودع الكود المصدري والاتصال الآمن { display-mode: "form" }
PROJECT_NAME = "MySystem" #@param {type:"string"}
GIT_SOURCE_URL = "https://github.com/octocat/Hello-World.git" #@param {type:"string"}
SOURCE_AUTH_METHOD = "HTTPS" #@param ["HTTPS", "SSH"]

if os.path.exists('/content/drive/MyDrive'):
    BASE_WORKSPACE = Path('/content/drive/MyDrive/AI_Documentation_Workspace')
else:
    BASE_WORKSPACE = Path('/content/AI_Documentation_Workspace')

from mdaar_pipeline import MDAARPipeline, ModelGateway, DualModelManager

pipeline = MDAARPipeline(
    project_name=PROJECT_NAME,
    base_workspace=BASE_WORKSPACE,
    git_source_url=GIT_SOURCE_URL,
    source_auth=SOURCE_AUTH_METHOD
)

if SOURCE_AUTH_METHOD == "SSH":
    source_pub_key = pipeline.setup_source_ssh()
    print('='*70)
    print('🔑 مفتاح SSH العام لمستودع الكود (Source Deploy Key):')
    print('أضفه في إعدادات مستودع الكود (Settings -> Deploy Keys / Access Keys):')
    print(source_pub_key)
    print('='*70)

print('⏳ جاري فحص الاتصال بمستودع الكود المصدري...')
source_conn = pipeline.test_source_connection()
print(source_conn['message'])
if not source_conn['success'] and SOURCE_AUTH_METHOD == "SSH":
    print('💡 أضف مفتاح الـ Deploy Key أعلاه في مستودعك ثم أعد تشغيل هذه الخلية للتحقق من نجاح الاتصال.')

print(f'✅ مجلد المشروع جاهز في المسار:\n📁 {pipeline.project_dir}')
print(f'🔍 مسار فهرس RAG الهجين: {pipeline.rag_index_dir}')

## 3️⃣ ضبط مستودع التوثيق الخارجي المستقل (Docs Repo - اختياري)
للحفظ المركزي في مستودع توثيق منفصل برفع آمن عبر مفتاح SSH مستقل أو HTTPS.

In [ ]:
#@title 3. ضبط مستودع التوثيق المنفصل والمفتاح المستقل { display-mode: "form" }
ENABLE_EXTERNAL_DOCS_REPO = False #@param {type:"boolean"}
GIT_DOCS_URL = "" #@param {type:"string"}
DOCS_AUTH_METHOD = "HTTPS" #@param ["HTTPS", "SSH"]

if ENABLE_EXTERNAL_DOCS_REPO and GIT_DOCS_URL.strip():
    pipeline.git_docs_url = GIT_DOCS_URL.strip()
    pipeline.docs_auth = DOCS_AUTH_METHOD
    if DOCS_AUTH_METHOD == "SSH":
        docs_pub_key = pipeline.setup_docs_ssh()
        print('='*70)
        print('🔑 مفتاح SSH العام لمستودع التوثيق (Docs Deploy Key — مع صلاحية Write):')
        print(docs_pub_key)
        print('='*70)
    print('⏳ جاري فحص الاتصال بمستودع التوثيق...')
    docs_conn = pipeline.test_docs_connection()
    print(docs_conn['message'])
    if not docs_conn['success'] and DOCS_AUTH_METHOD == "SSH":
        print('💡 أضف المفتاح أعلاه في مستودع التوثيق بصلاحية Write ثم أعد تشغيل هذه الخلية لتأكيد الاتصال.')
else:
    pipeline.git_docs_url = None
    print('ℹ️ التوثيق وفهرس RAG سيُحفظان محلياً في مساحة العمل (Drive/Local).')

## 4️⃣ البنية المزدوجة لنماذج الذكاء الاصطناعي (إعدادات مودل الخطة + إعدادات مودل التنفيذ)
يمكنك الآن توزيع العمل بذكاء أو تشغيل نموذج واحد لكليهما:
- **إعدادات مودل الخطة (Planner Model):** يبني الخطة الهندسية الشاملة للمشروع ويراجع جودة المخرجات (يُنصح هنا بـ Gemini أو نموذج استنتاجي قوي).
- **إعدادات مودل التنفيذ (Executor Model):** يتولى دائماً **فحص 100% من ملفات المشروع (`Map Pass`)** وكتابة تفاصيل الوثائق والرد عبر RAG (أو اختر «نفس مودل الخطة» لاستخدام نموذج واحد فقط).
> 💡 في حال اختيار نموذجين محليين مختلفين على كارت Colab T4 (15GB)، يقوم `DualModelManager` تلقائياً بتفريغ النموذج الخامل من ذاكرة الـ VRAM قبل تحميل الآخر لمنع الانهيار.

In [ ]:
#@title 4أ. إعدادات مودل الخطة (Planner Model Settings) { display-mode: "form" }
PLANNER_PROVIDER = "Gemini" #@param ["Gemini", "Local-Offline"]
PLANNER_GEMINI_MODEL = "gemini-2.5-flash" #@param ["gemini-2.5-flash", "gemini-1.5-pro", "gemini-2.0-flash"]
PLANNER_GEMINI_KEYS_INPUT = "" #@param {type:"string"}
PLANNER_LOCAL_MODEL_NAME = "Qwen/Qwen2.5-Coder-7B-Instruct" #@param ["Qwen/Qwen2.5-Coder-7B-Instruct", "meta-llama/Meta-Llama-3.1-8B-Instruct", "mistralai/Mistral-7B-Instruct-v0.3", "google/gemma-2-9b-it", "Qwen/Qwen2.5-7B-Instruct", "Qwen/Qwen2.5-Coder-1.5B-Instruct"]
PLANNER_QUANTIZATION = "4bit" #@param ["4bit", "None"]
PLANNER_HF_TOKEN_INPUT = "" #@param {type:"string"}

def _resolve_colab_secret(name):
    try:
        from google.colab import userdata
        return userdata.get(name)
    except Exception:
        return None

planner_keys = [k.strip() for k in PLANNER_GEMINI_KEYS_INPUT.split(',') if k.strip()]
if not planner_keys:
    sec_gem = _resolve_colab_secret('GEMINI_API_KEY')
    if sec_gem:
        planner_keys.append(sec_gem)

planner_hf = PLANNER_HF_TOKEN_INPUT.strip() or _resolve_colab_secret('HF_TOKEN')

planner_gateway = ModelGateway(
    provider=PLANNER_PROVIDER,
    model_name=PLANNER_GEMINI_MODEL,
    api_keys=planner_keys,
    local_model_path=PLANNER_LOCAL_MODEL_NAME,
    hf_token=planner_hf,
    quantization=PLANNER_QUANTIZATION
)
pipeline.planner_gateway = planner_gateway
pipeline.model_manager = DualModelManager(pipeline.planner_gateway, pipeline.executor_gateway)

print(f'✅ تم ضبط مودل الخطة: {planner_gateway.descriptor}')

In [ ]:
#@title 4ب. إعدادات مودل التنفيذ (Executor Model Settings — يفحص الكود ويكتب الوثائق) { display-mode: "form" }
EXECUTOR_MODE = "نفس مودل الخطة (مودل واحد)" #@param ["نفس مودل الخطة (مودل واحد)", "مخصص (مودل مستقل للتنفيذ والفحص)"]
EXECUTOR_PROVIDER = "Local-Offline" #@param ["Local-Offline", "Gemini"]
EXECUTOR_GEMINI_MODEL = "gemini-2.5-flash" #@param ["gemini-2.5-flash", "gemini-2.0-flash", "gemini-1.5-pro"]
EXECUTOR_GEMINI_KEYS_INPUT = "" #@param {type:"string"}
EXECUTOR_LOCAL_MODEL_NAME = "Qwen/Qwen2.5-Coder-7B-Instruct" #@param ["Qwen/Qwen2.5-Coder-7B-Instruct", "meta-llama/Meta-Llama-3.1-8B-Instruct", "mistralai/Mistral-7B-Instruct-v0.3", "google/gemma-2-9b-it", "Qwen/Qwen2.5-7B-Instruct", "Qwen/Qwen2.5-Coder-1.5B-Instruct"]
EXECUTOR_QUANTIZATION = "4bit" #@param ["4bit", "None"]
EXECUTOR_HF_TOKEN_INPUT = "" #@param {type:"string"}

if EXECUTOR_MODE == "نفس مودل الخطة (مودل واحد)":
    executor_gateway = pipeline.planner_gateway
    print(f'ℹ️ وضع النموذج الواحد: مودل التنفيذ هو نفسه مودل الخطة ({executor_gateway.descriptor}).')
else:
    exec_keys = [k.strip() for k in EXECUTOR_GEMINI_KEYS_INPUT.split(',') if k.strip()]
    if not exec_keys:
        sec_gem = _resolve_colab_secret('GEMINI_API_KEY') if '_resolve_colab_secret' in globals() else None
        if sec_gem:
            exec_keys.append(sec_gem)
    exec_hf = EXECUTOR_HF_TOKEN_INPUT.strip() or (_resolve_colab_secret('HF_TOKEN') if '_resolve_colab_secret' in globals() else None)
    executor_gateway = ModelGateway(
        provider=EXECUTOR_PROVIDER,
        model_name=EXECUTOR_GEMINI_MODEL,
        api_keys=exec_keys,
        local_model_path=EXECUTOR_LOCAL_MODEL_NAME,
        hf_token=exec_hf,
        quantization=EXECUTOR_QUANTIZATION
    )
    print(f'✅ تم ضبط مودل التنفيذ المستقل (للفحص والكتابة): {executor_gateway.descriptor}')

pipeline.executor_gateway = executor_gateway
pipeline.gateway = executor_gateway
pipeline.model_manager = DualModelManager(pipeline.planner_gateway, pipeline.executor_gateway)
print(f'🧠 الخطة: {pipeline.planner_gateway.descriptor}  |  ⚙️ التنفيذ والفحص: {pipeline.executor_gateway.descriptor}')

## 5️⃣ استنساخ الكود والتحقق الحتمي قبل البدء
تزامن المستودع مع فشل فوري آمن إذا تعذر الاستنساخ أو كان المستودع فارغاً (لمنع هلوسة النماذج).

In [ ]:
#@title 5. مزامنة الكود ورصد الملفات المعدلة { display-mode: "form" }
current_commit, changed_files = pipeline.sync_source_repo()

print(f'📌 حالة الـ Commit الحالي: ({current_commit[:7] if current_commit else "Local"})')
if changed_files:
    print(f'🔍 تم رصد {len(changed_files)} ملفاً متغيراً منذ آخر توثيق:')
    for f in changed_files[:15]:
        print('  •', f)
else:
    print('ℹ️ سيتم إجراء فحص كامل ومباشر لجميع ملفات المشروع.')

## 6️⃣ الفحص الكامل 100%، الخطة، التوليد المتعدد، بوابة الجودة وفهرس RAG
يبدأ دائماً بفحص كامل وجديد لجميع الملفات (دون الاعتماد على كاش جلسات سابقة منقطعة)، ثم يبني خطة التنفيذ الشاملة ويكتب كل وثائق الحزم الأربع ويفحص جودتها ويحفظ فهرس RAG الدائم في `.rag_index/`.

In [ ]:
#@title 6. تنفيذ الفحص الكامل 100% وتوليد جميع حزم التوثيق { display-mode: "form" }
FORCE_FULL_GENERATION = True #@param {type:"boolean"}

def _show_map_progress(idx, total, module_name, files_desc):
    if idx == 1 or idx == total or idx % 5 == 0:
        print(f'  [{idx}/{total}] فحص الوحدة `{module_name}`: {files_desc[:90]}')

target_changes = None if FORCE_FULL_GENERATION else changed_files
run_stats = pipeline.run_multi_agent_generation(
    changed_files=target_changes,
    force_full=FORCE_FULL_GENERATION,
    progress_callback=_show_map_progress
)

if current_commit:
    pipeline.save_state(current_commit)

print('='*70)
print('📊 ملخص دورة التوثيق المكتملة:')
print(json.dumps(run_stats, indent=2, ensure_ascii=False))
print('='*70)

## 7️⃣ إطلاق بوابة التوثيق التفاعلية (MkDocs Material + Cloudflare Tunnel)
بناء الموقع التفاعلي ومشاركته عبر نفق عام، مع رفع الوثائق لمستودع التوثيق الخارجي إن كان مفعلاً.

In [ ]:
#@title 7. تشغيل البوابة ومزامنة مستودع الوثائق { display-mode: "form" }
tunnel_url = pipeline.serve_and_tunnel(port=8000)

if tunnel_url:
    print('='*70)
    print('🎉 بوابة التوثيق التفاعلية جاهزة ومتاحة عبر الإنترنت:')
    print(f'👉 {tunnel_url}')
    print('='*70)
else:
    print('⚠️ جاري تشغيل الخادم محلياً، يمكنك التحقق من سجلات النفق.')

sync_docs_res = pipeline.sync_docs_repo()
print('📡 مزامنة مستودع الوثائق:', sync_docs_res)

## 8️⃣ المساعد المعرفي الذكي الهجين (Hybrid BM25 RAG — `.rag_index/`)
يسترجع من الوثائق المولّدة + بطاقات الملفات الحقيقية + جداول البيانات + نقاط الـ API المحفوظة في `project_dir/.rag_index/`، مع عزل مستوى الجمهور (`Developer`, `Management`, `End-User`).

In [ ]:
#@title 8. اسأل مساعد التوثيق الذكي حول المشروع { display-mode: "form" }
QUESTION = "ما هي معمارية النظام والمكونات الرئيسية والجداول والواجهات الفعلية؟" #@param {type:"string"}
AUDIENCE = "Developer" #@param ["Developer", "Management", "End-User"]

chat_engine = pipeline.get_chat_engine()
chunks_loaded = chat_engine.load_documents(prefer_saved_index=True)
print(f'📚 تم تحميل {chunks_loaded} مقطعاً معرفياً من `{chat_engine.index_dir}`')
print(f'🔍 جاري الاسترجاع والإجابة عن: "{QUESTION}" (الجمهور: {AUDIENCE})...\n')

answer = chat_engine.ask(QUESTION, audience=AUDIENCE)
print('='*70)
print(answer)
print('='*70)